# ☁️ Cloud Security — Real-Time Anomaly Detection + GenAI Breach Minimisation

### Complete Pipeline
| Step | Description |
|------|-------------|
| 1 | Install & Import |
| 2 | Dataset Upload & Auto-Detection |
| 3 | EDA & Visualisation |
| 4 | Feature Engineering |
| 5 | Isolation Forest — 4 Hyperparameter Configs Compared |
| 6 | Full Evaluation (Precision / Recall / F1 / AUC / ROC / Confusion Matrix) |
| 7 | Real-Time Anomaly Scoring |
| 8 | Risk Triage — CRITICAL / HIGH / MEDIUM Severity Banding |
| 9 | GenAI RAG Breach-Minimisation (Claude API) |
| 10 | Interactive Dashboard |
| 11 | Save All Artefacts |

---
> **⚠️ Before running:**
> 1. Upload `cloud_security_dataset.csv` using the 📁 Files panel (left sidebar)
> 2. Add your Anthropic key: left sidebar → 🔑 **Secrets** → Name: `ANTHROPIC_API_KEY`
> 3. Then click **Runtime → Run all**

## Step 1 · Install & Import

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Install
# ─────────────────────────────────────────────────────────────
import subprocess, sys
pkgs = ['anthropic', 'scikit-learn', 'pandas', 'numpy',
        'matplotlib', 'seaborn', 'joblib', 'ipywidgets']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=False)

# ─────────────────────────────────────────────────────────────
#  Imports
# ─────────────────────────────────────────────────────────────
import os, json, textwrap, warnings, time, re
from io import StringIO
from datetime import datetime

import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import seaborn               as sns

from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.ensemble        import IsolationForest
from sklearn.decomposition   import PCA
from sklearn.metrics         import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    ConfusionMatrixDisplay
)

import joblib
import anthropic

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi'      : 120,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'font.family'     : 'monospace',
    'axes.grid'       : True,
    'grid.alpha'      : 0.25,
})

# ─────────────────────────────────────────────────────────────
#  Colab check
# ─────────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'✅  Libraries loaded  |  Colab={IN_COLAB}')

## Step 2 · Dataset Upload & Auto-Column Detection

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Universal dataset loader — works with ANY CSV
#  Automatically detects: label col, numeric cols, cat cols
# ─────────────────────────────────────────────────────────────

def smart_load(path: str) -> pd.DataFrame:
    """Load CSV with automatic encoding and separator detection."""
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
    raise ValueError(f'Cannot read {path}')


def auto_detect_columns(df: pd.DataFrame) -> dict:
    """
    Automatically identify label, numeric, and categorical columns.
    Pre-trained heuristics for cloud-security datasets.
    """
    info = {}

    # --- label / severity column
    label_keywords = ['severity', 'label', 'attack', 'threat', 'class',
                       'anomaly', 'category', 'type', 'incident_type']
    for kw in label_keywords:
        match = [c for c in df.columns if kw in c.lower()]
        if match:
            info['label_col'] = match[0]
            break
    if 'label_col' not in info:
        info['label_col'] = df.columns[-1]   # fallback: last column

    # --- numeric features
    info['num_cols'] = df.select_dtypes(include='number').columns.tolist()

    # --- categorical features
    info['cat_cols'] = df.select_dtypes(include='object').columns.tolist()

    # --- timestamp column (if any)
    ts_kw = ['year', 'date', 'time', 'timestamp']
    info['ts_col'] = next(
        (c for c in df.columns if any(k in c.lower() for k in ts_kw)), None
    )

    # --- mitigation / effectiveness
    mit_kw = ['mitigation', 'effectiveness', 'score', 'rating']
    info['mit_col'] = next(
        (c for c in df.select_dtypes(include='number').columns
         if any(k in c.lower() for k in mit_kw)), None
    )

    return info


# ── Load the dataset ─────────────────────────────────────────
CSV_FILE = 'cloud_security_dataset.csv'

if not os.path.exists(CSV_FILE):
    if IN_COLAB:
        from google.colab import files
        print('📂  File not found. Please upload cloud_security_dataset.csv:')
        uploaded = files.upload()
        CSV_FILE = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError(
            f'{CSV_FILE} not found. Place the CSV in the same directory.'
        )

df_raw = smart_load(CSV_FILE)
META   = auto_detect_columns(df_raw)

print('\n📦  Dataset loaded')
print(f'    Shape         : {df_raw.shape}')
print(f'    Columns       : {list(df_raw.columns)}')
print(f'    Detected label: {META["label_col"]}')
print(f'    Numeric cols  : {META["num_cols"]}')
print(f'    Categorical   : {META["cat_cols"]}')
print(f'    Timestamp col : {META["ts_col"]}')
print(f'    Mitigation col: {META["mit_col"]}')
df_raw.head()

## Step 3 · EDA & Visualisation

In [ ]:
print('=== Dataset Overview ===')
print(f'Rows                : {len(df_raw):,}')
print(f'Columns             : {len(df_raw.columns)}')
print(f'Missing values      : {df_raw.isnull().sum().sum()}')
print(f'Duplicate rows      : {df_raw.duplicated().sum()}')
print()
print(df_raw.describe(include='all').T.to_string())

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('☁️  Cloud Security Dataset — EDA Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

THREAT_COLORS = [
    '#6366f1','#f97316','#22c55e','#ef4444','#eab308',
    '#06b6d4','#a855f7','#f43f5e','#84cc16','#0ea5e9'
]
SEV_PAL = {'High':'#ef4444', 'Medium':'#f97316', 'Low':'#22c55e'}

# ── Plot 1: Threat category bar
ax1 = fig.add_subplot(gs[0, 0])
tc  = df_raw['Threat_Category'].value_counts()
bars = ax1.barh(tc.index, tc.values, color=THREAT_COLORS[:len(tc)], edgecolor='white', lw=0.4)
ax1.set_title('Threat Category Distribution', fontweight='bold')
ax1.set_xlabel('Count')
for bar, val in zip(bars, tc.values):
    ax1.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=8)

# ── Plot 2: Severity pie
ax2 = fig.add_subplot(gs[0, 1])
sev = df_raw['Severity'].value_counts()
ax2.pie(sev.values, labels=sev.index,
        colors=[SEV_PAL.get(s,'#94a3b8') for s in sev.index],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':1.5})
ax2.set_title('Severity Distribution', fontweight='bold')

# ── Plot 3: Layer donut
ax3 = fig.add_subplot(gs[0, 2])
lyr = df_raw['Layer'].value_counts()
wedges, texts, autotexts = ax3.pie(
    lyr.values, labels=lyr.index,
    colors=['#6366f1','#f97316','#22c55e'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'width':0.55, 'edgecolor':'white', 'linewidth':1.5}
)
ax3.set_title('Cloud Layer Distribution', fontweight='bold')

# ── Plot 4: Incidents over years
ax4 = fig.add_subplot(gs[1, 0])
yearly = df_raw.groupby('Year')['Incident_Count'].sum()
ax4.plot(yearly.index, yearly.values, color='#f97316', lw=2.5, marker='o', ms=4)
ax4.fill_between(yearly.index, yearly.values, alpha=0.15, color='#f97316')
ax4.set_title('Total Incidents per Year', fontweight='bold')
ax4.set_xlabel('Year'); ax4.set_ylabel('Total')

# ── Plot 5: Mitigation Effectiveness by Severity
ax5 = fig.add_subplot(gs[1, 1])
for sev_val, color in SEV_PAL.items():
    subset = df_raw[df_raw['Severity'] == sev_val]['Mitigation_Effectiveness']
    ax5.hist(subset, bins=20, alpha=0.65, color=color, label=sev_val, edgecolor='white', lw=0.3)
ax5.set_title('Mitigation Effectiveness by Severity', fontweight='bold')
ax5.set_xlabel('Score (1–10)'); ax5.legend(fontsize=8)

# ── Plot 6: Mean Incident Count per Threat × Layer heatmap
ax6 = fig.add_subplot(gs[1, 2])
pivot = df_raw.pivot_table(
    values='Incident_Count', index='Threat_Category',
    columns='Layer', aggfunc='mean'
).round(1)
sns.heatmap(pivot, ax=ax6, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.3, cbar_kws={'label':'Mean Incidents', 'shrink':0.8})
ax6.set_title('Mean Incidents: Threat × Layer', fontweight='bold')
ax6.set_ylabel('')

# ── Plot 7: Incident_Count boxplot by Severity
ax7 = fig.add_subplot(gs[2, 0])
sev_order = ['Low','Medium','High']
data_box = [df_raw[df_raw['Severity']==s]['Incident_Count'].values for s in sev_order]
bp = ax7.boxplot(data_box, labels=sev_order, patch_artist=True,
                 medianprops={'color':'white','lw':2})
for patch, color in zip(bp['boxes'], [SEV_PAL[s] for s in sev_order]):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax7.set_title('Incident Count by Severity', fontweight='bold')
ax7.set_ylabel('Incident Count')

# ── Plot 8: Reported Cases distribution
ax8 = fig.add_subplot(gs[2, 1])
ax8.hist(df_raw['Reported_Cases'], bins=30, color='#6366f1',
         edgecolor='white', lw=0.4, alpha=0.85)
ax8.set_title('Reported Cases Distribution', fontweight='bold')
ax8.set_xlabel('Reported Cases')

# ── Plot 9: Mitigation vs Incident Count scatter
ax9 = fig.add_subplot(gs[2, 2])
for sev_val, color in SEV_PAL.items():
    sub = df_raw[df_raw['Severity'] == sev_val]
    ax9.scatter(sub['Mitigation_Effectiveness'], sub['Incident_Count'],
                alpha=0.25, s=10, color=color, label=sev_val)
ax9.set_title('Mitigation vs Incident Count', fontweight='bold')
ax9.set_xlabel('Mitigation Effectiveness')
ax9.set_ylabel('Incident Count')
ax9.legend(fontsize=8)

plt.savefig('eda_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅  EDA dashboard saved → eda_dashboard.png')

## Step 4 · Feature Engineering & Pre-Processing

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Pre-trained anomaly label logic
#  This logic is 'pre-trained' on domain knowledge of cloud
#  security incidents — covering all 10 threat types:
#
#  HIGH-severity events with LOW mitigation effectiveness are
#  the most dangerous unmitigated breaches. Additionally,
#  certain threat categories inherently represent attacks
#  (DDoS, Data Breach, Zero-Day, Container Escape, Supply Chain)
#  and are flagged even at Medium severity if mitigation is poor.
# ─────────────────────────────────────────────────────────────

HIGH_RISK_THREATS = {
    'DDoS Attack', 'Data Breach', 'Zero-Day Vulnerability',
    'Container Escape', 'Supply Chain Compromise'
}

def build_anomaly_label(row):
    sev  = row['Severity']
    mit  = row['Mitigation_Effectiveness']
    cat  = row['Threat_Category']
    inc  = row['Incident_Count']
    # Rule 1: High severity + poor mitigation
    if sev == 'High' and mit < 5:
        return 1
    # Rule 2: High-risk threat type + Medium severity + poor mitigation
    if cat in HIGH_RISK_THREATS and sev == 'Medium' and mit < 3.5:
        return 1
    # Rule 3: Extremely high incident count (>75) regardless of mitigation
    if inc > 75 and mit < 4:
        return 1
    return 0

df = df_raw.copy()
df['is_anomaly'] = df.apply(build_anomaly_label, axis=1)

print('Anomaly label distribution:')
print(df['is_anomaly'].value_counts())
print(f'\nAnomaly rate: {df["is_anomaly"].mean()*100:.2f} %')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Feature engineering
# ─────────────────────────────────────────────────────────────

df_enc = df.copy()

# 1. Ordinal encode Severity (preserves order: Low < Medium < High)
df_enc['Severity_enc'] = df_enc['Severity'].map({'Low': 0, 'Medium': 1, 'High': 2})

# 2. Label encode Threat_Category and Layer
le_threat = LabelEncoder()
le_layer  = LabelEncoder()
df_enc['Threat_enc'] = le_threat.fit_transform(df_enc['Threat_Category'])
df_enc['Layer_enc']  = le_layer.fit_transform(df_enc['Layer'])

# 3. Engineered features
df_enc['risk_score']          = df_enc['Severity_enc'] * (10 - df_enc['Mitigation_Effectiveness'])
df_enc['incident_per_case']   = df_enc['Incident_Count'] / (df_enc['Reported_Cases'] + 1)
df_enc['year_recency']        = df_enc['Year'] - df_enc['Year'].min()
df_enc['low_mitigation_flag'] = (df_enc['Mitigation_Effectiveness'] < 4).astype(int)
df_enc['high_incident_flag']  = (df_enc['Incident_Count'] > 70).astype(int)

FEATURE_COLS = [
    'year_recency',
    'Threat_enc',
    'Layer_enc',
    'Severity_enc',
    'Incident_Count',
    'Mitigation_Effectiveness',
    'Reported_Cases',
    'risk_score',
    'incident_per_case',
    'low_mitigation_flag',
    'high_incident_flag',
]

X_raw = df_enc[FEATURE_COLS].values
y     = df_enc['is_anomaly'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Feature matrix : {X_scaled.shape}')
print(f'Features used  : {FEATURE_COLS}')
print(f'Anomaly count  : {y.sum():,} / {len(y):,}  ({y.mean()*100:.2f} %)')

In [ ]:
# Feature correlation heatmap
feat_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
feat_df['is_anomaly'] = y

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.zeros_like(feat_df.corr())
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    feat_df.corr().round(2), annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.3, cbar_kws={'shrink': 0.75},
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('feature_correlation.png', bbox_inches='tight', dpi=130)
plt.show()

## Step 5 · Isolation Forest — 4 Hyperparameter Configurations

| Config | `n_estimators` | `max_samples` | `contamination` | `max_features` | Notes |
|--------|---------------|---------------|-----------------|----------------|-------|
| **Baseline** | 100 | `'auto'` | 0.09 | 1.0 | sklearn default, contamination = true anomaly rate |
| **High-Tree** | 300 | 512 | 0.09 | 1.0 | More trees → lower variance, stabler scores |
| **Tuned** | 200 | 0.8 | 0.07 | 0.8 | Sub-sampled features → diversity between trees |
| **Robust** | 300 | 1.0 | 0.11 | 0.6 | Full sample, aggressive contamination → max recall |

In [ ]:
TRUE_CONTAMINATION = round(float(y.mean()), 3)

CONFIGS = {
    'Baseline'  : dict(n_estimators=100, max_samples='auto',
                       contamination=TRUE_CONTAMINATION,
                       max_features=1.0, random_state=42),
    'High-Tree' : dict(n_estimators=300, max_samples=512,
                       contamination=TRUE_CONTAMINATION,
                       max_features=1.0, random_state=42),
    'Tuned'     : dict(n_estimators=200, max_samples=0.8,
                       contamination=max(0.01, TRUE_CONTAMINATION - 0.02),
                       max_features=0.8, random_state=42),
    'Robust'    : dict(n_estimators=300, max_samples=1.0,
                       contamination=min(0.5, TRUE_CONTAMINATION + 0.02),
                       max_features=0.6, random_state=42),
}

results = {}

for name, params in CONFIGS.items():
    t0 = time.time()
    print(f'⏳  [{name}] training ...', end=' ', flush=True)

    model  = IsolationForest(**params)
    model.fit(X_scaled)

    raw_pred = model.predict(X_scaled)            # +1 = normal, -1 = anomaly
    y_pred   = (raw_pred == -1).astype(int)       # 1 = anomaly
    scores   = -model.decision_function(X_scaled) # higher = more anomalous

    prec = precision_score(y, y_pred, zero_division=0)
    rec  = recall_score   (y, y_pred, zero_division=0)
    f1   = f1_score       (y, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y, scores)
    except Exception:
        auc = np.nan

    elapsed = time.time() - t0

    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'scores'   : scores,
        'Precision': round(prec, 4),
        'Recall'   : round(rec,  4),
        'F1'       : round(f1,   4),
        'AUC-ROC'  : round(auc,  4),
        'Time_s'   : round(elapsed, 2),
        'params'   : params,
    }
    print(f'Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  '
          f'AUC={auc:.3f}  ({elapsed:.1f}s)')

# Pick best by F1
BEST_CONFIG = max(results, key=lambda k: results[k]['F1'])
print(f'\n🏆  Best config: [{BEST_CONFIG}]  F1={results[BEST_CONFIG]["F1"]}')

## Step 6 · Full Evaluation & Comparison

In [ ]:
# ── Comparison table ──────────────────────────────────────────
compare_rows = []
for name, r in results.items():
    row = {
        'Config'        : name,
        'n_estimators'  : r['params']['n_estimators'],
        'max_samples'   : str(r['params']['max_samples']),
        'contamination' : r['params']['contamination'],
        'max_features'  : r['params']['max_features'],
        'Precision'     : r['Precision'],
        'Recall'        : r['Recall'],
        'F1-Score'      : r['F1'],
        'AUC-ROC'       : r['AUC-ROC'],
        'Train_Time(s)' : r['Time_s'],
    }
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows).set_index('Config')

print('\n' + '='*70)
print('         HYPERPARAMETER COMPARISON TABLE')
print('='*70)
display(
    compare_df.style
    .highlight_max(subset=['Precision','Recall','F1-Score','AUC-ROC'], color='#bbf7d0')
    .highlight_min(subset=['Precision','Recall','F1-Score','AUC-ROC'], color='#fecaca')
    .format(precision=4)
    .set_caption('🟢 Best  🔴 Worst per metric')
)

In [ ]:
# ── Metric bar charts ─────────────────────────────────────────
CHART_COLORS = ['#6366f1','#f97316','#22c55e','#f43f5e']
metrics      = ['Precision','Recall','F1','AUC-ROC']
config_names = list(results.keys())

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Isolation Forest — Hyperparameter Comparison',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, metrics):
    vals = [results[c][metric] for c in config_names]
    bars = ax.bar(config_names, vals, color=CHART_COLORS, width=0.55,
                  edgecolor='white', linewidth=0.6)
    ax.set_ylim(0, 1.12)
    ax.set_title(metric, fontweight='bold')
    ax.set_xticklabels(config_names, rotation=20, ha='right', fontsize=9)
    best_val = max(vals)
    for bar, val in zip(bars, vals):
        color = '#166534' if val == best_val else '#374151'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.015,
                f'{val:.3f}', ha='center', fontsize=9,
                fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('if_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── Confusion matrices — all configs ─────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Confusion Matrices — All Configurations',
             fontsize=13, fontweight='bold')

for ax, (name, r) in zip(axes, results.items()):
    cm   = confusion_matrix(y, r['y_pred'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Normal','Anomaly']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── ROC curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for (name, r), color in zip(results.items(), CHART_COLORS):
    fpr, tpr, _ = roc_curve(y, r['scores'])
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{name}  AUC={r['AUC-ROC']:.3f}")

ax.plot([0,1],[0,1],'--', color='#9ca3af', lw=1.2)
ax.fill_between([0,1],[0,1], alpha=0.04, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Configurations',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── PCA 2D scatter — best model ───────────────────────────────
best_scores = results[BEST_CONFIG]['scores']
best_pred   = results[BEST_CONFIG]['y_pred']
best_model  = results[BEST_CONFIG]['model']

pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

idx_sample = np.random.choice(len(X_2d), min(4000, len(X_2d)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'PCA Feature Space — Best Config: [{BEST_CONFIG}]',
             fontweight='bold', fontsize=13)

# Ground truth
sc1 = axes[0].scatter(
    X_2d[idx_sample, 0], X_2d[idx_sample, 1],
    c=y[idx_sample], cmap='RdYlGn_r', s=8, alpha=0.5
)
axes[0].set_title('Ground Truth (Green=Normal, Red=Anomaly)', fontweight='bold')
plt.colorbar(sc1, ax=axes[0], label='is_anomaly')

# Anomaly score heatmap
sc2 = axes[1].scatter(
    X_2d[idx_sample, 0], X_2d[idx_sample, 1],
    c=best_scores[idx_sample], cmap='plasma', s=8, alpha=0.5
)
axes[1].set_title('IF Anomaly Score Heatmap', fontweight='bold')
plt.colorbar(sc2, ax=axes[1], label='Anomaly Score')

for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.tight_layout()
plt.savefig('pca_scatter.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── Feature importance (correlation with anomaly score) ───────
score_series = pd.Series(best_scores, name='anomaly_score')
feat_df2     = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
feat_imp     = feat_df2.corrwith(score_series).abs().sort_values()

fig, ax = plt.subplots(figsize=(8, max(4, len(feat_imp)*0.4)))
colors_imp = plt.cm.plasma(feat_imp.values / feat_imp.max())
ax.barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor='white', lw=0.3)
ax.set_xlabel('|Pearson Correlation with Anomaly Score|')
ax.set_title(f'Feature Relevance — [{BEST_CONFIG}]', fontweight='bold')
for i, val in enumerate(feat_imp.values):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=130)
plt.show()

TOP_FEATURES = feat_imp.tail(5).index.tolist()[::-1]
print(f'Top 5 anomaly drivers: {TOP_FEATURES}')

## Step 7 · Real-Time Anomaly Scoring Engine

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Real-Time Scoring Engine
#  Accepts a single event dict or a DataFrame of new events.
#  Returns enriched predictions with scores + severity band.
# ─────────────────────────────────────────────────────────────

def encode_new_event(event_dict: dict) -> np.ndarray:
    """
    Encode a raw event dictionary into the feature vector
    expected by the trained model. Handles missing fields gracefully.
    """
    # Default values
    defaults = {
        'Year': 2024,
        'Threat_Category': 'DDoS Attack',
        'Layer': 'IaaS',
        'Severity': 'Medium',
        'Incident_Count': 55,
        'Mitigation_Effectiveness': 5.0,
        'Reported_Cases': 1,
    }
    defaults.update(event_dict)
    e = defaults

    sev_map = {'Low': 0, 'Medium': 1, 'High': 2}

    try:
        threat_enc = int(le_threat.transform([e['Threat_Category']])[0])
    except Exception:
        threat_enc = 0
    try:
        layer_enc  = int(le_layer.transform([e['Layer']])[0])
    except Exception:
        layer_enc  = 0

    sev_enc = sev_map.get(e['Severity'], 1)
    mit     = float(e['Mitigation_Effectiveness'])
    inc     = float(e['Incident_Count'])
    rep     = float(e['Reported_Cases'])
    year_r  = float(e['Year']) - df['Year'].min()

    feat_vec = np.array([[
        year_r,
        threat_enc,
        layer_enc,
        sev_enc,
        inc,
        mit,
        rep,
        sev_enc * (10 - mit),     # risk_score
        inc / (rep + 1),          # incident_per_case
        int(mit < 4),             # low_mitigation_flag
        int(inc > 70),            # high_incident_flag
    ]], dtype=float)

    return scaler.transform(feat_vec)


def score_event(event_dict: dict, model=None) -> dict:
    """
    Score a single event in real-time.
    Returns prediction, anomaly score, and severity band.
    """
    if model is None:
        model = best_model

    X_new  = encode_new_event(event_dict)
    pred   = model.predict(X_new)[0]        # +1 or -1
    score  = float(-model.decision_function(X_new)[0])

    is_anomaly = pred == -1

    # Severity band based on score percentile benchmarks from training
    p67 = np.percentile(best_scores[best_pred == 1], 67) if best_pred.sum() > 0 else 0.3
    p90 = np.percentile(best_scores[best_pred == 1], 90) if best_pred.sum() > 0 else 0.6

    if is_anomaly:
        if score >= p90:
            band = 'CRITICAL'
        elif score >= p67:
            band = 'HIGH'
        else:
            band = 'MEDIUM'
    else:
        band = 'NORMAL'

    return {
        'is_anomaly'    : is_anomaly,
        'anomaly_score' : round(score, 4),
        'severity_band' : band,
        'raw_prediction': int(pred),
    }


# ── Demo: score 4 test events ─────────────────────────────────
TEST_EVENTS = [
    {'Year':2024,'Threat_Category':'DDoS Attack',   'Layer':'IaaS','Severity':'High',  'Incident_Count':82,'Mitigation_Effectiveness':2.1,'Reported_Cases':5},
    {'Year':2023,'Threat_Category':'Data Breach',   'Layer':'SaaS','Severity':'High',  'Incident_Count':75,'Mitigation_Effectiveness':1.8,'Reported_Cases':3},
    {'Year':2022,'Threat_Category':'Insider Threat','Layer':'PaaS','Severity':'Medium','Incident_Count':45,'Mitigation_Effectiveness':7.5,'Reported_Cases':1},
    {'Year':2021,'Threat_Category':'API Exploit',   'Layer':'PaaS','Severity':'Low',   'Incident_Count':35,'Mitigation_Effectiveness':9.0,'Reported_Cases':1},
]

print('\n=== REAL-TIME SCORING — TEST EVENTS ===')
print(f'{"Event":<5} {"Threat":<28} {"Sev":<8} {"Score":<8} {"Band":<10} {"Anomaly"}')
print('-'*75)
for i, ev in enumerate(TEST_EVENTS):
    out = score_event(ev)
    flag = '🚨' if out['is_anomaly'] else '✅'
    print(f'{i+1:<5} {ev["Threat_Category"]:<28} {ev["Severity"]:<8} '
          f'{out["anomaly_score"]:<8.4f} {out["severity_band"]:<10} {flag}')

print('\n✅  Real-time scoring engine ready.')

## Step 8 · Risk Triage — Severity Banding

In [ ]:
# Attach scores and predictions to full dataframe
df_result = df.copy()
df_result['anomaly_score'] = best_scores
df_result['if_prediction'] = best_pred

# Only flagged anomalies
anomaly_df = df_result[df_result['if_prediction'] == 1].copy()

# Percentile-based banding
p67 = np.percentile(anomaly_df['anomaly_score'], 67)
p90 = np.percentile(anomaly_df['anomaly_score'], 90)

def assign_band(score):
    if score >= p90: return 'CRITICAL'
    if score >= p67: return 'HIGH'
    return 'MEDIUM'

anomaly_df['severity_band'] = anomaly_df['anomaly_score'].apply(assign_band)

print('=== RISK TRIAGE SUMMARY ===')
print(anomaly_df['severity_band'].value_counts().to_string())
print(f'\nTotal flagged events : {len(anomaly_df):,}')
print(f'CRITICAL threshold   : anomaly_score >= {p90:.4f}')
print(f'HIGH threshold       : anomaly_score >= {p67:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Risk Triage — Anomaly Severity Dashboard',
             fontweight='bold', fontsize=13)

BAND_COLORS = {'CRITICAL':'#ef4444','HIGH':'#f97316','MEDIUM':'#eab308'}

# 1. Band counts
bc = anomaly_df['severity_band'].value_counts()
bars = axes[0].bar(bc.index, bc.values,
                   color=[BAND_COLORS[b] for b in bc.index],
                   edgecolor='white', lw=0.5)
axes[0].set_title('Events per Severity Band', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, bc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, str(val),
                 ha='center', fontweight='bold')

# 2. Threat category by band
tc_band = anomaly_df.groupby(['Threat_Category','severity_band']).size().unstack(fill_value=0)
tc_band.plot(kind='barh', ax=axes[1], stacked=True,
             color=[BAND_COLORS.get(c,'#94a3b8') for c in tc_band.columns],
             edgecolor='white', lw=0.3)
axes[1].set_title('Threat Category × Severity Band', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].legend(title='Band', fontsize=8)

# 3. Score distribution per band
for band, color in BAND_COLORS.items():
    subset = anomaly_df[anomaly_df['severity_band'] == band]['anomaly_score']
    axes[2].hist(subset, bins=20, alpha=0.65, color=color,
                 label=band, edgecolor='white', lw=0.3)
axes[2].axvline(p67, color='#f97316', ls='--', lw=1.5, label=f'p67={p67:.3f}')
axes[2].axvline(p90, color='#ef4444', ls='--', lw=1.5, label=f'p90={p90:.3f}')
axes[2].set_title('Anomaly Score Distribution by Band', fontweight='bold')
axes[2].set_xlabel('Anomaly Score')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('risk_triage.png', bbox_inches='tight', dpi=130)
plt.show()

## Step 9 · GenAI RAG Breach-Minimisation (Claude API)

### Architecture
```
Isolation Forest flags anomaly
        │
        ▼
Context Builder
  ├── Event features (real data)
  ├── MITRE ATT&CK knowledge base (RAG retrieval)
  └── Cloud threat playbooks (RAG retrieval)
        │  structured prompt
        ▼
 Claude claude-sonnet-4-20250514
  ├── Validate / correct attack classification
  ├── Root cause analysis
  ├── Immediate containment (3 actions)
  ├── Long-term hardening (3 actions)
  ├── Blast radius estimate
  └── SOC priority (P1 / P2 / P3)
        │
        ▼
 Structured JSON Playbook → Dashboard
```

**GenAI Techniques Used:**
| Technique | How it's used here |
|-----------|-------------------|
| **RAG** | MITRE + playbook KB injected per-event |
| **Structured Output** | Strict JSON schema enforced in prompt |
| **Chain-of-Thought** | Claude validates attack type before remediating |
| **Domain Grounding** | MITRE ATT&CK Cloud Matrix TTPs in every prompt |
| **Contextual Reasoning** | All 7 event fields provided for nuanced analysis |

In [ ]:
# ─────────────────────────────────────────────────────────────
#  KNOWLEDGE BASE — Pre-trained on all 10 threat categories
#  in this dataset. Acts as the RAG vector store.
# ─────────────────────────────────────────────────────────────

THREAT_KB = {
    'DDoS Attack': {
        'mitre_tactic'   : 'Impact (TA0040)',
        'mitre_technique': 'T1498 – Network Denial of Service',
        'risk_factors'   : ['High incident volume', 'IaaS layer exposure', 'Low mitigation'],
        'immediate'      : [
            'Enable AWS Shield Advanced / Azure DDoS Protection Standard immediately',
            'Configure rate-limiting rules on WAF and CDN edge (block >1000 req/s per IP)',
            'Scale auto-scaling group to absorb traffic; alert NOC',
        ],
        'hardening'      : [
            'Deploy anycast network diffusion across multiple PoPs',
            'Implement BGP blackholing for volumetric attacks',
            'Schedule quarterly DDoS simulation exercises',
        ],
    },
    'Data Breach': {
        'mitre_tactic'   : 'Exfiltration (TA0010)',
        'mitre_technique': 'T1048 – Exfiltration Over Alternative Protocol',
        'risk_factors'   : ['SaaS data exposure', 'Unencrypted data at rest', 'Weak DLP'],
        'immediate'      : [
            'Immediately revoke all active session tokens and IAM credentials in scope',
            'Quarantine affected storage buckets and enable object-level logging',
            'Notify DPO and initiate 72-hour GDPR/regulatory breach notification window',
        ],
        'hardening'      : [
            'Enable customer-managed encryption (CMK) for all storage resources',
            'Deploy cloud-native DLP with ML classification on egress traffic',
            'Implement zero-trust data access with attribute-based controls (ABAC)',
        ],
    },
    'Zero-Day Vulnerability': {
        'mitre_tactic'   : 'Initial Access (TA0001)',
        'mitre_technique': 'T1190 – Exploit Public-Facing Application',
        'risk_factors'   : ['No patch available', 'High CVSS score', 'Active exploitation'],
        'immediate'      : [
            'Isolate affected services behind WAF with virtual patching rules',
            'Enable enhanced logging and SIEM correlation for IOC patterns',
            'Activate incident response retainer; brief CISO and legal',
        ],
        'hardening'      : [
            'Subscribe to vendor security advisories and automate patch pipeline',
            'Implement runtime application self-protection (RASP)',
            'Deploy deception technology (honeypots) to detect lateral movement',
        ],
    },
    'API Exploit': {
        'mitre_tactic'   : 'Credential Access (TA0006)',
        'mitre_technique': 'T1552 – Unsecured Credentials / T1190 – API Abuse',
        'risk_factors'   : ['Exposed API keys', 'Missing rate limits', 'Broken object auth'],
        'immediate'      : [
            'Rotate all API keys and OAuth tokens immediately',
            'Enable API gateway throttling (max 100 req/min per key)',
            'Block suspicious API caller IPs via gateway ACL',
        ],
        'hardening'      : [
            'Implement OWASP API Top-10 controls across all endpoints',
            'Deploy API security posture management (APISPM) tooling',
            'Enforce mTLS for all service-to-service API communication',
        ],
    },
    'Insider Threat': {
        'mitre_tactic'   : 'Collection (TA0009)',
        'mitre_technique': 'T1078 – Valid Accounts / T1213 – Data from Info Repositories',
        'risk_factors'   : ['Privileged access abuse', 'Anomalous working hours', 'Data staging'],
        'immediate'      : [
            'Suspend account and force re-authentication with MFA challenge',
            'Preserve audit logs and initiate HR + legal hold procedure',
            'Review all data access events in past 30 days for staging behaviour',
        ],
        'hardening'      : [
            'Deploy User & Entity Behaviour Analytics (UEBA) for baseline deviation alerts',
            'Enforce just-in-time (JIT) privileged access with approval workflows',
            'Conduct background re-screening for high-privilege users annually',
        ],
    },
    'IAM Misconfiguration': {
        'mitre_tactic'   : 'Privilege Escalation (TA0004)',
        'mitre_technique': 'T1548 – Abuse Elevation Control Mechanism',
        'risk_factors'   : ['Overly permissive IAM policies', 'Wildcard permissions', 'No MFA'],
        'immediate'      : [
            'Run IAM Access Analyzer and revoke all overly-broad wildcard policies',
            'Force MFA on all human identities; disable all unused service accounts',
            'Enable CloudTrail in all regions with integrity validation',
        ],
        'hardening'      : [
            'Adopt least-privilege IAM with permission boundaries',
            'Implement continuous CSPM scanning (Prisma Cloud / Wiz / Defender)',
            'Enforce Service Control Policies (SCPs) at AWS Org level',
        ],
    },
    'Container Escape': {
        'mitre_tactic'   : 'Privilege Escalation (TA0004)',
        'mitre_technique': 'T1611 – Escape to Host',
        'risk_factors'   : ['Privileged containers', 'Host PID/network sharing', 'No runtime security'],
        'immediate'      : [
            'Terminate the affected container and drain the node immediately',
            'Cordon the Kubernetes node and isolate it from cluster traffic',
            'Forensic snapshot of node filesystem for IOC extraction',
        ],
        'hardening'      : [
            'Enforce Pod Security Standards (restricted profile) cluster-wide',
            'Deploy eBPF-based runtime security (Falco / Tetragon)',
            'Use distroless or scratch base images; scan with Trivy in CI/CD',
        ],
    },
    'Encryption Failure': {
        'mitre_tactic'   : 'Defense Evasion (TA0005)',
        'mitre_technique': 'T1600 – Weaken Encryption',
        'risk_factors'   : ['Data exposed in transit', 'Weak cipher suites', 'Key mismanagement'],
        'immediate'      : [
            'Enforce TLS 1.3 minimum on all load balancers and API gateways',
            'Rotate KMS keys and re-encrypt affected data stores',
            'Audit all S3 bucket policies for public-read ACLs',
        ],
        'hardening'      : [
            'Implement certificate lifecycle automation (ACM / cert-manager)',
            'Deploy HSM-backed key management for all encryption keys',
            'Enable server-side encryption by default on all cloud storage',
        ],
    },
    'Supply Chain Compromise': {
        'mitre_tactic'   : 'Initial Access (TA0001)',
        'mitre_technique': 'T1195 – Supply Chain Compromise',
        'risk_factors'   : ['Compromised dependencies', 'Malicious packages', 'CI/CD poisoning'],
        'immediate'      : [
            'Pin all dependency versions and audit SBOMs for known-malicious hashes',
            'Rotate all CI/CD secrets and pipeline credentials immediately',
            'Block the compromised package at artifact repository level',
        ],
        'hardening'      : [
            'Adopt SLSA Level 3 supply chain security framework',
            'Integrate SCA (Snyk / FOSSA) into every pull request',
            'Use a private artifact registry with allowlist for approved packages',
        ],
    },
    'Compliance Violation': {
        'mitre_tactic'   : 'Exfiltration / Impact (TA0010 / TA0040)',
        'mitre_technique': 'T1537 – Transfer Data to Cloud Account',
        'risk_factors'   : ['Regulatory non-compliance', 'Audit failure', 'Data residency breach'],
        'immediate'      : [
            'Notify compliance officer and legal; initiate breach assessment',
            'Enable AWS Config / Azure Policy remediation for non-compliant resources',
            'Suspend data transfers to non-approved regions immediately',
        ],
        'hardening'      : [
            'Deploy continuous compliance monitoring (Prowler / Scout Suite)',
            'Implement data residency controls with cloud guardrails',
            'Schedule quarterly third-party compliance audits',
        ],
    },
}

print(f'✅  Knowledge base loaded: {len(THREAT_KB)} threat categories')
print('    Covers:', list(THREAT_KB.keys()))

In [ ]:
# ─────────────────────────────────────────────────────────────
#  GenAI helper functions
# ─────────────────────────────────────────────────────────────

RESPONSE_SCHEMA = '''
{
  "confirmed_attack_type"   : string,
  "confidence"              : "LOW|MEDIUM|HIGH",
  "root_cause_summary"      : string (max 50 words),
  "mitre_tactic"            : string,
  "mitre_technique"         : string,
  "immediate_actions"       : [string, string, string],
  "long_term_hardening"     : [string, string, string],
  "estimated_blast_radius"  : "LOW|MEDIUM|HIGH|CRITICAL",
  "soc_priority"            : "P1|P2|P3",
  "monitoring_kpis"         : [string, string]
}
'''


def build_rag_prompt(event: dict, kb_entry: dict) -> str:
    """
    Build a RAG-style prompt:
    - Retrieves relevant KB entry for the threat type (Retrieval)
    - Injects event data + KB into prompt (Augmentation)
    - Claude reasons over both  (Generation)
    """
    event_str = '\n'.join(
        f'  {k}: {v}' for k, v in event.items()
        if k not in ['if_prediction', 'is_anomaly']
    )
    kb_str = json.dumps(kb_entry, indent=2)

    return textwrap.dedent(f"""
    You are an expert cloud security analyst with deep knowledge of MITRE ATT&CK,
    AWS/Azure/GCP security services, and incident response playbooks.

    An Isolation Forest ML model has flagged the following cloud security event as
    an anomaly requiring immediate attention.

    ## Flagged Event (real sensor data)
    {event_str}

    ## Retrieved Threat Intelligence (RAG context)
    {kb_str}

    ## Instructions
    1. Analyse the event data carefully against the threat intelligence.
    2. Validate or correct the attack classification using chain-of-thought reasoning.
    3. Consider the Mitigation_Effectiveness score (low = attacker has foothold).
    4. Produce a complete remediation playbook as valid JSON matching this exact schema:

    {RESPONSE_SCHEMA}

    IMPORTANT: Return ONLY the JSON object. No markdown fences, no explanation outside JSON.
    """).strip()


def call_claude_api(prompt: str, api_key: str) -> dict:
    """Call Claude claude-sonnet-4-20250514 and parse structured JSON."""
    client  = anthropic.Anthropic(api_key=api_key)
    message = client.messages.create(
        model      = 'claude-sonnet-4-20250514',
        max_tokens = 1200,
        messages   = [{'role': 'user', 'content': prompt}],
    )
    raw = message.content[0].text.strip()
    # Strip markdown code fences if model adds them
    raw = re.sub(r'^```[a-zA-Z]*\n?', '', raw)
    raw = re.sub(r'\n?```$', '', raw)
    return json.loads(raw.strip())


def mock_claude_response(event: dict, kb_entry: dict) -> dict:
    """Deterministic fallback when no API key is present."""
    band = event.get('severity_band', 'HIGH')
    mit  = float(event.get('Mitigation_Effectiveness', 5))
    return {
        'confirmed_attack_type'  : event.get('Threat_Category', 'Unknown'),
        'confidence'             : 'HIGH' if mit < 3 else 'MEDIUM',
        'root_cause_summary'     : (
            f"Anomalous {event.get('Threat_Category','threat')} activity detected on "
            f"{event.get('Layer','cloud')} layer. Mitigation score {mit:.1f}/10 indicates "
            f"{'active attacker foothold' if mit < 4 else 'partial control'}."
        ),
        'mitre_tactic'           : kb_entry.get('mitre_tactic', 'Unknown'),
        'mitre_technique'        : kb_entry.get('mitre_technique', 'Unknown'),
        'immediate_actions'      : kb_entry.get('immediate', ['Investigate immediately'])[:3],
        'long_term_hardening'    : kb_entry.get('hardening',  ['Implement security baseline'])[:3],
        'estimated_blast_radius' : 'CRITICAL' if band == 'CRITICAL' else 'HIGH' if band == 'HIGH' else 'MEDIUM',
        'soc_priority'           : 'P1' if band == 'CRITICAL' else 'P2' if band == 'HIGH' else 'P3',
        'monitoring_kpis'        : [
            'Mean time to detect (MTTD) < 5 minutes',
            'Mean time to respond (MTTR) < 30 minutes',
        ],
    }


def run_genai_remediation(event_row: dict, api_key: str = '') -> dict:
    """Full RAG pipeline for a single event."""
    threat  = event_row.get('Threat_Category', 'DDoS Attack')
    kb_entry = THREAT_KB.get(threat, list(THREAT_KB.values())[0])

    if api_key:
        try:
            prompt   = build_rag_prompt(event_row, kb_entry)
            response = call_claude_api(prompt, api_key)
        except Exception as e:
            print(f'    ⚠️  API error ({e}) — using mock response')
            response = mock_claude_response(event_row, kb_entry)
    else:
        response = mock_claude_response(event_row, kb_entry)

    return response


print('✅  GenAI pipeline functions defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Load API key from Colab Secrets
# ─────────────────────────────────────────────────────────────

ANTHROPIC_API_KEY = ''

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

USE_LIVE_API = bool(ANTHROPIC_API_KEY and ANTHROPIC_API_KEY.startswith('sk-'))

if USE_LIVE_API:
    print('✅  Anthropic API key found — will use live Claude claude-sonnet-4-20250514')
else:
    print('⚠️  No API key found — running with mock responses')
    print('    To enable: Secrets (🔑) → ANTHROPIC_API_KEY → your key')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Process top CRITICAL + HIGH anomalies through GenAI
# ─────────────────────────────────────────────────────────────

# Select top events: all CRITICAL + top 5 HIGH
critical_events = anomaly_df[anomaly_df['severity_band'] == 'CRITICAL'].nlargest(5, 'anomaly_score')
high_events     = anomaly_df[anomaly_df['severity_band'] == 'HIGH'].nlargest(3, 'anomaly_score')
top_events      = pd.concat([critical_events, high_events]).head(8)

remediation_results = []

print(f'Processing {len(top_events)} events through GenAI RAG pipeline...')
print('─'*70)

for idx, row in top_events.iterrows():
    event_dict = row.to_dict()
    threat     = event_dict.get('Threat_Category', 'Unknown')
    band       = event_dict.get('severity_band', 'HIGH')
    score      = event_dict.get('anomaly_score', 0)

    print(f'\n🔍  Event #{idx}  |  {threat}  |  Band={band}  |  Score={score:.4f}')

    response = run_genai_remediation(event_dict, ANTHROPIC_API_KEY)

    response['event_idx']        = idx
    response['severity_band']    = band
    response['anomaly_score']    = round(score, 4)
    response['Threat_Category']  = threat
    response['Layer']            = event_dict.get('Layer', '')
    response['Severity']         = event_dict.get('Severity', '')
    response['Mitigation_Score'] = event_dict.get('Mitigation_Effectiveness', '')

    remediation_results.append(response)

    print(f'    ✓ Confirmed type   : {response["confirmed_attack_type"]}')
    print(f'    ✓ Confidence       : {response["confidence"]}')
    print(f'    ✓ SOC Priority     : {response["soc_priority"]}')
    print(f'    ✓ Blast Radius     : {response["estimated_blast_radius"]}')
    print(f'    ✓ MITRE            : {response["mitre_technique"]}')
    print(f'    ✓ Top action       : {response["immediate_actions"][0]}')

print(f'\n✅  GenAI remediation complete — {len(remediation_results)} playbooks generated')

In [ ]:
# ── Remediation summary table ─────────────────────────────────
rem_df = pd.DataFrame([{
    'Event #'          : r['event_idx'],
    'Threat'           : r['Threat_Category'],
    'Layer'            : r['Layer'],
    'Band'             : r['severity_band'],
    'Score'            : r['anomaly_score'],
    'Confidence'       : r['confidence'],
    'MITRE Tactic'     : r['mitre_tactic'],
    'Blast Radius'     : r['estimated_blast_radius'],
    'SOC Priority'     : r['soc_priority'],
    'Top Action'       : r['immediate_actions'][0][:60] + '...' if len(r['immediate_actions'][0]) > 60 else r['immediate_actions'][0],
} for r in remediation_results])

print('\n===== GENAI REMEDIATION SUMMARY =====')

def color_band(val):
    colors = {'CRITICAL':'background-color:#fecaca','HIGH':'background-color:#fed7aa',
              'MEDIUM':'background-color:#fef08a','LOW':'background-color:#bbf7d0'}
    return colors.get(val, '')

def color_soc(val):
    colors = {'P1':'background-color:#fecaca;font-weight:bold',
              'P2':'background-color:#fed7aa','P3':'background-color:#fef08a'}
    return colors.get(val, '')

display(
    rem_df.style
    .applymap(color_band, subset=['Band','Blast Radius'])
    .applymap(color_soc,  subset=['SOC Priority'])
    .set_caption('GenAI Playbook Summary — Claude claude-sonnet-4-20250514')
)

In [ ]:
# Print full playbooks for CRITICAL events
print('\n' + '='*70)
print('  FULL REMEDIATION PLAYBOOKS — CRITICAL EVENTS')
print('='*70)

for r in remediation_results:
    if r['severity_band'] != 'CRITICAL':
        continue
    print(f"""
╔══════════════════════════════════════════════════════════════════════
║  EVENT #{r['event_idx']}  |  {r['Threat_Category']}  |  {r['Layer']}  |  {r['severity_band']}
╠══════════════════════════════════════════════════════════════════════
║  Confirmed Type    : {r['confirmed_attack_type']}
║  Confidence        : {r['confidence']}
║  Anomaly Score     : {r['anomaly_score']}
║  Blast Radius      : {r['estimated_blast_radius']}
║  SOC Priority      : {r['soc_priority']}
║  MITRE Tactic      : {r['mitre_tactic']}
║  MITRE Technique   : {r['mitre_technique']}
╠══════════════════════════════════════════════════════════════════════
║  ROOT CAUSE
║  {r['root_cause_summary']}
╠══════════════════════════════════════════════════════════════════════
║  IMMEDIATE ACTIONS (execute now)
║  1. {r['immediate_actions'][0]}
║  2. {r['immediate_actions'][1] if len(r['immediate_actions'])>1 else 'N/A'}
║  3. {r['immediate_actions'][2] if len(r['immediate_actions'])>2 else 'N/A'}
╠══════════════════════════════════════════════════════════════════════
║  LONG-TERM HARDENING
║  1. {r['long_term_hardening'][0]}
║  2. {r['long_term_hardening'][1] if len(r['long_term_hardening'])>1 else 'N/A'}
║  3. {r['long_term_hardening'][2] if len(r['long_term_hardening'])>2 else 'N/A'}
╠══════════════════════════════════════════════════════════════════════
║  MONITORING KPIs
║  • {r['monitoring_kpis'][0] if r['monitoring_kpis'] else 'MTTD < 5 min'}
║  • {r['monitoring_kpis'][1] if len(r.get('monitoring_kpis',[]))>1 else 'MTTR < 30 min'}
╚══════════════════════════════════════════════════════════════════════
""")

In [ ]:
# ── GenAI output visualisation ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('GenAI Breach Minimisation — Output Analysis',
             fontweight='bold', fontsize=13)

BAND_C = {'CRITICAL':'#ef4444','HIGH':'#f97316','MEDIUM':'#eab308','LOW':'#22c55e'}
SOC_C  = {'P1':'#ef4444','P2':'#f97316','P3':'#eab308'}

# Blast radius
br = rem_df['Blast Radius'].value_counts()
axes[0].pie(
    br.values, labels=br.index,
    colors=[BAND_C.get(b,'#94a3b8') for b in br.index],
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':1.5}
)
axes[0].set_title('Estimated Blast Radius', fontweight='bold')

# SOC priority
sp = rem_df['SOC Priority'].value_counts().sort_index()
bars = axes[1].bar(
    sp.index, sp.values,
    color=[SOC_C.get(p,'#94a3b8') for p in sp.index],
    edgecolor='white', lw=0.5
)
axes[1].set_title('SOC Priority Distribution', fontweight='bold')
axes[1].set_ylabel('Events')
for bar, val in zip(bars, sp.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05, str(val),
                 ha='center', fontweight='bold')

# MITRE tactic distribution
mt = rem_df['MITRE Tactic'].value_counts()
axes[2].barh(mt.index, mt.values, color='#6366f1', edgecolor='white', lw=0.3)
axes[2].set_title('MITRE Tactic Breakdown', fontweight='bold')
axes[2].set_xlabel('Events')

plt.tight_layout()
plt.savefig('genai_analysis.png', bbox_inches='tight', dpi=130)
plt.show()

## Step 10 · Interactive Real-Time Dashboard

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Interactive Dashboard using ipywidgets
#  Enter any cloud event → instant IF scoring + GenAI playbook
# ─────────────────────────────────────────────────────────────

import ipywidgets as widgets
from IPython.display import display as ipy_display, HTML, clear_output

# ── Widget definitions ────────────────────────────────────────
w_threat = widgets.Dropdown(
    options=sorted(df['Threat_Category'].unique()),
    value='DDoS Attack',
    description='Threat:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
w_layer = widgets.Dropdown(
    options=['IaaS','PaaS','SaaS'],
    value='IaaS',
    description='Layer:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
w_severity = widgets.Dropdown(
    options=['Low','Medium','High'],
    value='High',
    description='Severity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
w_year = widgets.IntSlider(
    value=2024, min=2010, max=2025, step=1,
    description='Year:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
w_incident = widgets.IntSlider(
    value=75, min=29, max=87, step=1,
    description='Incident Count:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
w_mitigation = widgets.FloatSlider(
    value=2.5, min=1.0, max=10.0, step=0.1,
    description='Mitigation Eff.:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px'),
    readout_format='.1f'
)
w_cases = widgets.IntSlider(
    value=3, min=1, max=21, step=1,
    description='Reported Cases:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
w_genai = widgets.Checkbox(
    value=True,
    description='Generate AI Playbook',
    indent=False
)
w_btn = widgets.Button(
    description='🔍  Analyse Event',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px')
)
w_out = widgets.Output()

# ── Layout ────────────────────────────────────────────────────
header = HTML("""
<div style="background:#18181b;padding:16px 20px;border-radius:10px;
            border-left:4px solid #f97316;margin-bottom:12px">
  <h2 style="color:#f97316;margin:0;font-family:monospace;font-size:16px">
    ☁️ Cloud Security — Real-Time Anomaly Analyser
  </h2>
  <p style="color:#a1a1aa;margin:4px 0 0;font-family:monospace;font-size:12px">
    Powered by Isolation Forest + Claude claude-sonnet-4-20250514 RAG
  </p>
</div>
""")

form = widgets.VBox([
    header,
    widgets.HBox([w_threat, w_layer, w_severity]),
    w_year, w_incident, w_mitigation, w_cases,
    widgets.HBox([w_genai, w_btn]),
    w_out,
])

# ── On-click handler ─────────────────────────────────────────
def on_analyse(btn):
    w_out.clear_output()
    with w_out:
        event = {
            'Year'                    : w_year.value,
            'Threat_Category'         : w_threat.value,
            'Layer'                   : w_layer.value,
            'Severity'                : w_severity.value,
            'Incident_Count'          : w_incident.value,
            'Mitigation_Effectiveness': w_mitigation.value,
            'Reported_Cases'          : w_cases.value,
        }

        # Score with IF
        result = score_event(event, best_model)

        BAND_EMOJI = {'CRITICAL':'🔴','HIGH':'🟠','MEDIUM':'🟡','NORMAL':'🟢'}
        SOC_COLOR  = {'P1':'#ef4444','P2':'#f97316','P3':'#eab308'}
        band       = result['severity_band']
        emoji      = BAND_EMOJI.get(band, '⚪')

        ipy_display(HTML(f"""
        <div style="background:#18181b;padding:16px;border-radius:10px;
                    border:1px solid #27272a;margin:8px 0;font-family:monospace">
          <h3 style="color:white;margin:0 0 12px">{emoji} IF Detection Result</h3>
          <table style="width:100%;border-collapse:collapse">
            <tr>
              <td style="color:#a1a1aa;padding:4px 8px">Prediction</td>
              <td style="color:{'#ef4444' if result['is_anomaly'] else '#22c55e'};
                          font-weight:bold;padding:4px 8px">
                {'⚠️  ANOMALY DETECTED' if result['is_anomaly'] else '✅  NORMAL TRAFFIC'}
              </td>
            </tr>
            <tr>
              <td style="color:#a1a1aa;padding:4px 8px">Anomaly Score</td>
              <td style="color:white;padding:4px 8px">{result['anomaly_score']:.4f}</td>
            </tr>
            <tr>
              <td style="color:#a1a1aa;padding:4px 8px">Severity Band</td>
              <td style="color:white;padding:4px 8px">{band}</td>
            </tr>
          </table>
        </div>
        """))

        if result['is_anomaly'] and w_genai.value:
            event['severity_band'] = band
            print('⏳  Querying Claude for remediation playbook...')
            playbook = run_genai_remediation(event, ANTHROPIC_API_KEY)

            soc_color = SOC_COLOR.get(playbook['soc_priority'], '#94a3b8')
            imm = ''.join(f'<li style="margin:4px 0">{a}</li>' for a in playbook['immediate_actions'])
            hrd = ''.join(f'<li style="margin:4px 0">{a}</li>' for a in playbook['long_term_hardening'])
            kpi = ''.join(f'<li style="margin:4px 0">{k}</li>' for k in playbook.get('monitoring_kpis', []))

            ipy_display(HTML(f"""
            <div style="background:#18181b;padding:16px;border-radius:10px;
                        border:1px solid #f97316;margin:8px 0;font-family:monospace">
              <h3 style="color:#f97316;margin:0 0 12px">🤖 GenAI Remediation Playbook</h3>

              <div style="display:grid;grid-template-columns:1fr 1fr;
                          gap:8px;margin-bottom:12px">
                <div style="background:#27272a;padding:10px;border-radius:8px">
                  <div style="color:#71717a;font-size:11px">CONFIRMED TYPE</div>
                  <div style="color:white;font-weight:bold">{playbook['confirmed_attack_type']}</div>
                </div>
                <div style="background:#27272a;padding:10px;border-radius:8px">
                  <div style="color:#71717a;font-size:11px">CONFIDENCE</div>
                  <div style="color:white;font-weight:bold">{playbook['confidence']}</div>
                </div>
                <div style="background:#27272a;padding:10px;border-radius:8px">
                  <div style="color:#71717a;font-size:11px">SOC PRIORITY</div>
                  <div style="color:{soc_color};font-weight:bold">{playbook['soc_priority']}</div>
                </div>
                <div style="background:#27272a;padding:10px;border-radius:8px">
                  <div style="color:#71717a;font-size:11px">BLAST RADIUS</div>
                  <div style="color:#ef4444;font-weight:bold">{playbook['estimated_blast_radius']}</div>
                </div>
              </div>

              <div style="background:#1c1c1e;padding:10px;border-radius:8px;margin-bottom:8px">
                <div style="color:#71717a;font-size:11px;margin-bottom:4px">MITRE</div>
                <div style="color:#a78bfa">{playbook['mitre_technique']}</div>
              </div>

              <div style="background:#1c1c1e;padding:10px;border-radius:8px;margin-bottom:8px">
                <div style="color:#71717a;font-size:11px;margin-bottom:4px">ROOT CAUSE</div>
                <div style="color:#e4e4e7">{playbook['root_cause_summary']}</div>
              </div>

              <div style="background:#1c1c1e;padding:10px;border-radius:8px;margin-bottom:8px">
                <div style="color:#ef4444;font-size:11px;font-weight:bold;margin-bottom:6px">
                  ⚡ IMMEDIATE ACTIONS
                </div>
                <ul style="color:#e4e4e7;margin:0;padding-left:20px;font-size:13px">{imm}</ul>
              </div>

              <div style="background:#1c1c1e;padding:10px;border-radius:8px;margin-bottom:8px">
                <div style="color:#22c55e;font-size:11px;font-weight:bold;margin-bottom:6px">
                  🛡️  LONG-TERM HARDENING
                </div>
                <ul style="color:#e4e4e7;margin:0;padding-left:20px;font-size:13px">{hrd}</ul>
              </div>

              <div style="background:#1c1c1e;padding:10px;border-radius:8px">
                <div style="color:#6366f1;font-size:11px;font-weight:bold;margin-bottom:6px">
                  📊 MONITORING KPIs
                </div>
                <ul style="color:#e4e4e7;margin:0;padding-left:20px;font-size:13px">{kpi}</ul>
              </div>
            </div>
            """))

        elif not result['is_anomaly']:
            ipy_display(HTML("""
            <div style="background:#052e16;padding:12px 16px;border-radius:8px;
                        border-left:4px solid #22c55e;font-family:monospace">
              <span style="color:#22c55e;font-weight:bold">✅  Normal traffic pattern.</span>
              <span style="color:#86efac"> No remediation required.</span>
            </div>
            """))

w_btn.on_click(on_analyse)

print('✅  Dashboard ready — interact below:')
ipy_display(form)

## Step 11 · Final Summary & Save Artefacts

In [ ]:
print('\n' + '='*70)
print('       CLOUD SECURITY PIPELINE — COMPLETE SUMMARY')
print('='*70)
print(f'Dataset                  : {CSV_FILE}')
print(f'Total records            : {len(df):,}')
print(f'Features engineered      : {len(FEATURE_COLS)}')
print(f'Anomaly rate             : {y.mean()*100:.2f} %')
print(f'Anomalies detected       : {y.sum():,}')
print()
print('─── Isolation Forest Comparison ───')
for name, r in results.items():
    marker = ' ◀  BEST' if name == BEST_CONFIG else ''
    print(f'  [{name:10s}]  '
          f'Precision={r["Precision"]:.3f}  '
          f'Recall={r["Recall"]:.3f}  '
          f'F1={r["F1"]:.3f}  '
          f'AUC={r["AUC-ROC"]:.3f}  '
          f'Time={r["Time_s"]}s{marker}')
print()
print(f'Best Model               : {BEST_CONFIG}')
bp = results[BEST_CONFIG]['params']
print(f'  n_estimators           : {bp["n_estimators"]}')
print(f'  max_samples            : {bp["max_samples"]}')
print(f'  contamination          : {bp["contamination"]}')
print(f'  max_features           : {bp["max_features"]}')
print()
print('─── Risk Triage ───')
for band, count in anomaly_df['severity_band'].value_counts().items():
    print(f'  {band:10s}: {count:,} events')
print()
print('─── GenAI Remediation ───')
print(f'  Events processed       : {len(remediation_results)}')
print(f'  API mode               : {"Live Claude claude-sonnet-4-20250514" if USE_LIVE_API else "Mock (add ANTHROPIC_API_KEY to Secrets)"}')
print(f'  KB categories covered  : {len(THREAT_KB)}')
print()
print('='*70)

In [ ]:
# ── Save all artefacts ────────────────────────────────────────
import os

joblib.dump(best_model, 'best_if_model.pkl')
joblib.dump(scaler,     'scaler.pkl')
joblib.dump({
    'le_threat'    : le_threat,
    'le_layer'     : le_layer,
    'feature_cols' : FEATURE_COLS,
    'year_min'     : int(df['Year'].min()),
    'p67'          : float(p67),
    'p90'          : float(p90),
}, 'pipeline_meta.pkl')

anomaly_df.to_csv('anomaly_events.csv',       index=False)
rem_df.to_csv    ('remediation_playbook.csv', index=False)
compare_df.to_csv('if_comparison.csv')

print('Saved artefacts:')
saved = ['best_if_model.pkl','scaler.pkl','pipeline_meta.pkl',
         'anomaly_events.csv','remediation_playbook.csv','if_comparison.csv',
         'eda_dashboard.png','feature_correlation.png','if_comparison.png',
         'confusion_matrices.png','roc_curves.png','pca_scatter.png',
         'feature_importance.png','risk_triage.png','genai_analysis.png']

for f in saved:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  ✓  {f:<35} ({size:,} bytes)')

# Download from Colab
if IN_COLAB:
    from google.colab import files
    for f in ['remediation_playbook.csv','if_comparison.csv','best_if_model.pkl']:
        if os.path.exists(f):
            files.download(f)

print('\n🎉  Pipeline complete!')